In [0]:
%sql
-- ─── 1. PEDIDOS (SILVER) ──────────────────────────────────────
-- Transformações aplicadas vs Bronze:
--  • data_pedido / data_envio: DATE (já vem como date, mantém)
--  • cultura: INITCAP (padroniza capitalização)
--  • estado: UPPER (go → GO)
--  • localidade: TRIM + LOWER (padroniza)
--  • desconto: mantido como DECIMAL(5,4) — 0.1000 = 10%
--  • margem_liquida_pct: DECIMAL(6,2)
--  • safra: normalizada (remove espaços extras)
--  • categoria_margem: mantida (já é categórica limpa)
--  • lucro_justo_cepea: removido token Databricks que apareceu
--    na linha 3 do sample — campo deve ser DOUBLE, não STRING
 

In [0]:
%sql

-- Criando a tabela pedidos na camada Silver
-- ─── 1. PEDIDOS (SILVER) ──────────────────────────────────────
-- Transformações aplicadas vs Bronze:
--  • data_pedido / data_envio: DATE (já vem como date, mantém)
--  • cultura: INITCAP (padroniza capitalização)
--  • estado: UPPER (go → GO)
--  • localidade: TRIM + LOWER (padroniza)
--  • desconto: mantido como DECIMAL(5,4) — 0.1000 = 10%
--  • margem_liquida_pct: DECIMAL(6,2)
--  • safra: normalizada (remove espaços extras)
--  • categoria_margem: mantida (já é categórica limpa)
--  • lucro_justo_cepea: removido token Databricks que apareceu
--    na linha 3 do sample — campo deve ser DOUBLE, não STRING
 
CREATE TABLE workspace.gs_silver.pedidos (
  id_pedido               STRING        NOT NULL,
  data_pedido             DATE,
  data_envio              DATE,
  cultura                 STRING,
  cidade                  STRING,
  localidade              STRING,
  estado                  STRING,
  pais                    STRING,
  regiao                  STRING,
  subcultura              STRING,
  subcultura2             STRING,
  formato                 STRING,
  vendas                  DECIMAL(14,2),
  quantidade              DECIMAL(10,2),
  desconto                DECIMAL(5,4),
  lucro                   DECIMAL(14,2),
  custo_total             DECIMAL(14,2),
  margem_liquida_pct      DECIMAL(6,2),
  preco_por_unidade       DECIMAL(10,2),
  custo_por_unidade       DECIMAL(10,2),
  lucro_por_unidade       DECIMAL(10,2),
  ano                     INT,
  mes                     INT,
  trimestre               INT,
  safra                   STRING,
  categoria_margem        STRING,
  impacto_desconto        DECIMAL(14,2),
  custo_insumos_est       DECIMAL(14,2),
  custo_fertilizantes_est DECIMAL(14,2),
  custo_defensivos_est    DECIMAL(14,2),
  custo_outros_est        DECIMAL(14,2),
  cepea_referencia_sc     DECIMAL(10,2),
  gap_preco_cepea         DECIMAL(10,2),
  perc_vs_cepea           DECIMAL(8,4),
  receita_perdida_cepea   DECIMAL(14,2),
  lucro_justo_cepea       DECIMAL(14,2),
  preco_realizado_sc      DECIMAL(10,2),
  margem_bruta_pct        DECIMAL(8,2),
  updated_at              TIMESTAMP,
  source_file             STRING
)
USING DELTA;


In [0]:
%sql
select distinct(cultura) from workspace.gs_silver.pedidos

In [0]:
%sql
-- Criando a tabela calendário

-- ─── 2. CALENDARIO_SAFRAS (SILVER) ────────────────────────────
-- Transformações:
--  • Nulos em estagio/cor/datas: linhas sem estágio são meses
--    fora da janela do estágio — mantidas com NULL (correto)
--  • talhao: TRIM
--  • safra_tipo: LOWER TRIM
--  • mes_nome: padronizado (já vem curto 'Jan', 'Fev')
 
CREATE TABLE IF NOT EXISTS workspace.gs_silver.calendario_safras (
 
  cultura                 STRING,
  talhao                  STRING,
  ano                     INT,
  safra_tipo              STRING,       -- 'safrinha', 'safra_verao'
  safra_ano_label         STRING,
 
  mes_num                 INT,
  mes_nome                STRING,
  mes_ordem_exibicao      INT,
  mes_data_inicio         DATE,
  mes_data_fim            DATE,
 
  -- Estágio fenológico (NULL quando mês fora do estágio)
  estagio                 STRING,
  ordem_estagio           INT,
  cor_hex                 STRING,
  data_inicio_estagio     DATE,
  data_fim_estagio        DATE,
 
  -- Derivada: duração do estágio em dias
  duracao_estagio_dias    INT           GENERATED ALWAYS AS (
                            CASE WHEN data_inicio_estagio IS NOT NULL
                                  AND data_fim_estagio IS NOT NULL
                            THEN DATEDIFF(data_fim_estagio, data_inicio_estagio)
                            ELSE NULL END
                          ),
 
  -- Metadados
  updated_at              TIMESTAMP,
  source_file             STRING
)
USING DELTA
COMMENT 'Silver: calendário fenológico curado por cultura, talhão e safra';


In [0]:
%sql
-- Criando a tabela Saude Financeira

-- ─── 3. SAUDE_FINANCEIRA (SILVER) ────────────────────────────
-- Transformações:
--  • Todos os campos numéricos tipados como DECIMAL
--  • fazenda: LOWER TRIM
--  • Colunas derivadas: margem_liquida_pct, receita_por_ha, lucro_por_ha
--  • premissa: STRING (mantida para rastreabilidade do cenário)
 
CREATE TABLE IF NOT EXISTS workspace.gs_silver.saude_financeira (
 
  fazenda                             STRING,
  ano                                 INT,
 
  -- P&L
  receita_total                       DECIMAL(16,2),
  custo_total                         DECIMAL(16,2),
  lucro_total                         DECIMAL(16,2),
 
  -- Balanço
  passivo_circulante_base             DECIMAL(16,2),
  ativo_circulante_base               DECIMAL(16,2),
  capital_giro_base                   DECIMAL(16,2),
  liquidez_corrente_base              DECIMAL(6,2),
  area_ha                             INT,
 
  -- Simulação patrimonial
  ativo_total_simulado                DECIMAL(18,2),
  passivo_nao_circulante_simulado     DECIMAL(16,2),
  endividamento_geral_pct             DECIMAL(6,2),
 
  -- Simulação financeira
  despesas_financeiras_simuladas      DECIMAL(16,2),
  cobertura_juros_x                   DECIMAL(6,2),
 
  -- Score de risco
  score_risco_financeiro              INT,
  score_risco_classificacao           STRING,
 
  -- Premissa
  premissa                            STRING,
 
  -- Derivadas
  margem_liquida_pct                  DECIMAL(23,2)  GENERATED ALWAYS AS (
                                        CASE WHEN receita_total > 0
                                        THEN ROUND((lucro_total / receita_total) * 100, 2)
                                        ELSE NULL END
                                      ),
  receita_por_ha                      DECIMAL(17,2) GENERATED ALWAYS AS (
                                        CASE WHEN area_ha > 0
                                        THEN ROUND(receita_total / area_ha, 2)
                                        ELSE NULL END
                                      ),
  lucro_por_ha                        DECIMAL(17,2) GENERATED ALWAYS AS (
                                        CASE WHEN area_ha > 0
                                        THEN ROUND(lucro_total / area_ha, 2)
                                        ELSE NULL END
                                      ),
  custo_por_ha                        DECIMAL(17,2) GENERATED ALWAYS AS (
                                        CASE WHEN area_ha > 0
                                        THEN ROUND(custo_total / area_ha, 2)
                                        ELSE NULL END
                                      ),
 
  -- Metadados
  updated_at                          TIMESTAMP,
  source_file                         STRING
)
USING DELTA
COMMENT 'Silver: saúde financeira curada — CENÁRIO SIMULADO (CompreRural/Plano Safra/Ipiagri)'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
%sql
select distinct(*) from workspace.gs_silver.pedidos 

In [0]:
%sql
UPDATE workspace.gs_silver.pedidos
SET
    cultura          = LOWER(TRIM(cultura)),
    cidade           = LOWER(TRIM(cidade)),
    localidade       = LOWER(TRIM(localidade)),
    estado           = LOWER(TRIM(estado)),
    pais             = LOWER(TRIM(pais)),
    regiao           = LOWER(TRIM(regiao)),
    safra            = LOWER(TRIM(safra)),
    subcultura       = LOWER(TRIM(subcultura)),
    subcultura2      = LOWER(TRIM(subcultura2)),
    formato          = LOWER(TRIM(formato)),
    categoria_margem = LOWER(TRIM(categoria_margem)),
    source_file      = LOWER(TRIM(source_file));

In [0]:
%sql
UPDATE workspace.gs_silver.calendario_safras
SET
    cultura         = LOWER(TRIM(cultura)),
    talhao          = LOWER(TRIM(talhao)),
    safra_tipo      = LOWER(TRIM(safra_tipo)),
    safra_ano_label = LOWER(TRIM(safra_ano_label)),
    mes_nome        = LOWER(TRIM(mes_nome)),
    estagio         = LOWER(TRIM(estagio)),
    cor_hex         = LOWER(TRIM(cor_hex)),
    source_file     = LOWER(TRIM(source_file));

In [0]:
%sql
UPDATE workspace.gs_silver.saude_financeira
SET
    fazenda                   = LOWER(TRIM(fazenda)),
    score_risco_classificacao = LOWER(TRIM(score_risco_classificacao)),
    premissa                  = LOWER(TRIM(premissa)),
    source_file               = LOWER(TRIM(source_file));

In [0]:
%sql
drop table if exists workspace.gs_bronze.pragas_inseticidas_culturas

In [0]:
%sql
describe table workspace.gs_silver.pragas_defensivos_culturas

In [0]:
%sql
--select distinct(praga) from workspace.gs_silver.pragas_defensivos_culturas as s
describe table workspace.gs_silver.pragas_defensivos_culturas

In [0]:
%sql
SELECT f.id, f.nome, f.usuario_id, u.usuario
FROM workspace.gs_bronze.fazenda f
JOIN workspace.gs_bronze.usuario u ON u.id = f.usuario_id
ORDER BY f.id;

In [0]:
%sql
UPDATE workspace.gs_bronze.fazenda
SET usuario_id = '2'
WHERE nome = 'Fazenda Boa Esperança';

In [0]:
%sql
select * from workspace.gs_bronze.talhao

--delete from workspace.gs_bronze.talhao 


In [0]:
%sql
update workspace.gs_bronze.fazenda
set
id = 2
where nome = 'Fazenda Boa Esperança'

In [0]:
df = spark.table("workspace.gs_bronze.fazenda")
display(df)

In [0]:
%sql
select * from workspace.gs_bronze.talhao

In [0]:
%sql
SELECT id, COUNT(*) AS qtd
FROM workspace.gs_bronze.talhao
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY qtd DESC;